# AI TẠO SINH 2 – BỔ SUNG THÔNG TIN SUẤT CHIẾU CGV

## Mục tiêu
Notebook này kế thừa cách gọi Ollama trong `AI_tao_sinh.ipynb`, nhưng tạo thêm 3 thuộc tính số:
- `TicketPrice`: giá vé (VNĐ)
- `TotalSeats`: tổng số ghế của phòng chiếu
- `AvailableSeats`: số ghế còn trống

**Đầu ra:**
1. `AI_tao_sinh_2.csv`: bảng dữ liệu bổ sung do AI tạo sinh, chỉ chứa khóa suất chiếu và 3 cột mới.
2. `cgv_showtimes_AI_generated_v2.csv`: dataset lịch chiếu sau khi `merge` thêm 3 cột.

In [1]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Đặt notebook cùng thư mục với cgv_showtimes_AI_generated.csv
DATA_DIR = Path(".")
INPUT_FILE = DATA_DIR / "cgv_showtimes_AI_generated.csv"
OUTPUT_AI_FILE = DATA_DIR / "AI_tao_sinh_2.csv"
OUTPUT_MERGED_FILE = DATA_DIR / "cgv_showtimes_AI_generated_v2.csv"

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {INPUT_FILE}. Hãy đặt file CSV cùng thư mục với notebook."
    )

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print("Kích thước dataset:", df.shape)
print("Các cột:", df.columns.tolist())
display(df.head())

Kích thước dataset: (1172, 6)
Các cột: ['MovieName', 'Cinema', 'ShowDate', 'Format', 'ShowTime', 'GeneratedText']


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải nghiệm thú vị khi xem suất chiếu ""AVENGERS: HỒI KẾT..."
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Crescent Mall để xem phim CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI..."
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN LÃNG"" tại rạp chiếu CGV Crescent Mall và trải ngh..."


## 1. Kiểm tra các cột khóa

Mỗi dòng đại diện cho một suất chiếu. Vì vậy, không dùng riêng `MovieName` để ghép dữ liệu. Ta sử dụng các cột khóa gồm `MovieName`, `Cinema`, `ShowDate`, `Format`, `ShowTime` để phân biệt các suất chiếu.

In [2]:
KEY_COLUMNS = ["MovieName", "Cinema", "ShowDate", "Format", "ShowTime"]

missing_columns = [col for col in KEY_COLUMNS if col not in df.columns]
if missing_columns:
    raise KeyError(f"Dataset thiếu các cột khóa: {missing_columns}")

# Chuẩn hóa khóa dạng văn bản để hạn chế lỗi khoảng trắng khi merge
for col in KEY_COLUMNS:
    df[col] = df[col].astype("string").str.strip()

print("Số dòng:", len(df))
print("Số khóa suất chiếu duy nhất:", len(df.drop_duplicates(subset=KEY_COLUMNS)))

if df.duplicated(subset=KEY_COLUMNS).any():
    print("Cảnh báo: Có khóa suất chiếu bị trùng. Cần kiểm tra trước khi merge để tránh nhân bản dòng.")
    display(df[df.duplicated(subset=KEY_COLUMNS, keep=False)].sort_values(KEY_COLUMNS).head(10))
else:
    print("Không có khóa suất chiếu trùng.")

Số dòng: 1172
Số khóa suất chiếu duy nhất: 1172
Không có khóa suất chiếu trùng.


## 2. Tạo prompt cho Ollama

Yêu cầu Ollama chỉ trả về JSON với 3 trường số. Các giới hạn được kiểm tra lại bằng Python trước khi lưu.

In [3]:
def create_prompt(row):
    return f"""Bạn là hệ thống tạo dữ liệu mô phỏng cho một suất chiếu phim.

Thông tin suất chiếu:
- Tên phim: {row["MovieName"]}
- Rạp: {row["Cinema"]}
- Ngày chiếu: {row["ShowDate"]}
- Định dạng: {row["Format"]}
- Giờ chiếu: {row["ShowTime"]}

Hãy tạo dữ liệu mô phỏng và chỉ trả về một JSON hợp lệ có đúng 3 khóa:
{{
  "TicketPrice": số nguyên từ 50000 đến 130000 (đơn vị VNĐ),
  "TotalSeats": một trong các số 80, 100, 120,
  "AvailableSeats": số nguyên từ 0 đến TotalSeats
}}

Quy tắc:
- Chỉ trả về JSON, không markdown, không giải thích.
- Các giá trị phải là số nguyên, không có dấu phân cách hàng nghìn.
- AvailableSeats không được lớn hơn TotalSeats.
- Đây là dữ liệu mô phỏng, không phải dữ liệu thực tế của CGV.
"""


def generate_data(prompt):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {"temperature": 0.7}
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=180)
    response.raise_for_status()

    result = response.json()
    return json.loads(result["response"])

## 3. Gọi Ollama và kiểm tra dữ liệu

Notebook sẽ tạo một bản ghi cho từng suất chiếu. Nếu Ollama trả về dữ liệu không hợp lệ, chương trình thử lại tối đa 3 lần. Dòng không tạo được dữ liệu sẽ được ghi nhận để bạn kiểm tra, không tự ý điền thành dữ liệu thực.

In [4]:
generated_rows = []
failed_rows = []

for index, row in df.iterrows():
    prompt = create_prompt(row)
    generated = None

    for attempt in range(3):
        try:
            result = generate_data(prompt)

            ticket_price = int(result["TicketPrice"])
            total_seats = int(result["TotalSeats"])
            available_seats = int(result["AvailableSeats"])

            if not (50000 <= ticket_price <= 130000):
                raise ValueError("TicketPrice ngoài khoảng cho phép")

            if total_seats not in [80, 100, 120]:
                raise ValueError("TotalSeats không thuộc [80, 100, 120]")

            if not (0 <= available_seats <= total_seats):
                raise ValueError("AvailableSeats không hợp lệ")

            generated = {
                **{col: row[col] for col in KEY_COLUMNS},
                "TicketPrice": ticket_price,
                "AvailableSeats": available_seats,
                "TotalSeats": total_seats
            }
            break

        except Exception as e:
            print(f"Lỗi dòng {index}, lần thử {attempt + 1}: {e}")
            time.sleep(1)

    if generated is not None:
        generated_rows.append(generated)
    else:
        failed_rows.append(index)

    if (index + 1) % 25 == 0:
        print(f"Đã xử lý {index + 1}/{len(df)} dòng")

    time.sleep(0.2)

df_ai = pd.DataFrame(generated_rows)

print("Số dòng tạo thành công:", len(df_ai))
print("Số dòng thất bại:", len(failed_rows))
if failed_rows:
    print("Index thất bại:", failed_rows[:30])

display(df_ai.head())

Đã xử lý 25/1172 dòng
Đã xử lý 50/1172 dòng
Đã xử lý 75/1172 dòng
Đã xử lý 100/1172 dòng
Đã xử lý 125/1172 dòng
Đã xử lý 150/1172 dòng
Đã xử lý 175/1172 dòng
Đã xử lý 200/1172 dòng
Đã xử lý 225/1172 dòng
Đã xử lý 250/1172 dòng
Đã xử lý 275/1172 dòng
Đã xử lý 300/1172 dòng
Đã xử lý 325/1172 dòng
Đã xử lý 350/1172 dòng
Đã xử lý 375/1172 dòng
Đã xử lý 400/1172 dòng
Đã xử lý 425/1172 dòng
Đã xử lý 450/1172 dòng
Đã xử lý 475/1172 dòng
Đã xử lý 500/1172 dòng
Đã xử lý 525/1172 dòng
Đã xử lý 550/1172 dòng
Đã xử lý 575/1172 dòng
Đã xử lý 600/1172 dòng
Đã xử lý 625/1172 dòng
Đã xử lý 650/1172 dòng
Đã xử lý 675/1172 dòng
Đã xử lý 700/1172 dòng
Đã xử lý 725/1172 dòng
Đã xử lý 750/1172 dòng
Đã xử lý 775/1172 dòng
Đã xử lý 800/1172 dòng
Đã xử lý 825/1172 dòng
Lỗi dòng 840, lần thử 1: Expecting value: line 1 column 1 (char 0)
Đã xử lý 850/1172 dòng
Đã xử lý 875/1172 dòng
Đã xử lý 900/1172 dòng
Đã xử lý 925/1172 dòng
Đã xử lý 950/1172 dòng
Đã xử lý 975/1172 dòng
Đã xử lý 1000/1172 dòng
Đã xử lý 1025/1

,MovieName,Cinema,ShowDate,Format,ShowTime,TicketPrice,AvailableSeats,TotalSeats
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,70000,80,100
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,100000,90,100
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,90000,80,100
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,80000,90,100
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,100000,50,80


## 4. Lưu file AI_tao_sinh_2.csv

File này là bảng bổ sung độc lập, có thể dùng trong bước `merge` của Stage 1.

In [5]:
df_ai.to_csv(OUTPUT_AI_FILE, index=False, encoding="utf-8-sig")

print("Đã lưu:", OUTPUT_AI_FILE)
print("Kích thước file AI:", df_ai.shape)
print("Missing:")
display(df_ai.isna().sum().to_frame("missing"))

Đã lưu: AI_tao_sinh_2.csv
Kích thước file AI: (1172, 8)
Missing:


,missing
MovieName,0
Cinema,0
ShowDate,0
Format,0
ShowTime,0
TicketPrice,0
AvailableSeats,0
TotalSeats,0


## 5. Merge vào cgv_showtimes_AI_generated

Đây là phép **merge ngang** vì 3 cột mới mô tả chính các suất chiếu đã có. Không dùng `concat` để nối dọc hai bảng này.

Dùng `validate="one_to_one"` để Pandas báo lỗi nếu khóa không duy nhất và có nguy cơ làm nhân bản dòng.

In [6]:
# Kiểm tra khóa duy nhất ở cả hai bảng
if df.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Bảng lịch chiếu có khóa trùng. Hãy xử lý khóa trùng trước khi merge.")

if df_ai.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Bảng AI có khóa trùng. Không thể merge one_to_one.")

df_merged = df.merge(
    df_ai,
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one"
)

print("Kích thước trước merge:", df.shape)
print("Kích thước sau merge:", df_merged.shape)
print("Số dòng thiếu dữ liệu AI:")
display(df_merged[["TicketPrice", "AvailableSeats", "TotalSeats"]].isna().sum().to_frame("missing"))

df_merged.to_csv(OUTPUT_MERGED_FILE, index=False, encoding="utf-8-sig")
print("Đã lưu dataset mới:", OUTPUT_MERGED_FILE)
display(df_merged.head())

Kích thước trước merge: (1172, 6)
Kích thước sau merge: (1172, 9)
Số dòng thiếu dữ liệu AI:


,missing
TicketPrice,0
AvailableSeats,0
TotalSeats,0


Đã lưu dataset mới: cgv_showtimes_AI_generated_v2.csv


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText,TicketPrice,AvailableSeats,TotalSeats
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,70000,80,100
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải nghiệm thú vị khi xem suất chiếu ""AVENGERS: HỒI KẾT...",100000,90,100
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,90000,80,100
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Crescent Mall để xem phim CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI...",80000,90,100
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN LÃNG"" tại rạp chiếu CGV Crescent Mall và trải ngh...",100000,50,80


## 6. Kiểm tra tính hợp lệ sau merge

In [7]:
print("TicketPrice hợp lệ:",
      df_merged["TicketPrice"].dropna().between(75000, 150000).all())

print("AvailableSeats không âm:",
      (df_merged["AvailableSeats"].dropna() >= 0).all())

print("AvailableSeats <= TotalSeats:",
      (df_merged["AvailableSeats"].dropna() <=
       df_merged.loc[df_merged["AvailableSeats"].notna(), "TotalSeats"]).all())

print("Tổng số dòng giữ nguyên:", len(df_merged) == len(df))

display(df_merged[KEY_COLUMNS + [
    "TicketPrice", "AvailableSeats", "TotalSeats"
]].head(10))

TicketPrice hợp lệ: False
AvailableSeats không âm: True
AvailableSeats <= TotalSeats: True
Tổng số dòng giữ nguyên: True


,MovieName,Cinema,ShowDate,Format,ShowTime,TicketPrice,AvailableSeats,TotalSeats
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,70000,80,100
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,100000,90,100
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,90000,80,100
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,80000,90,100
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,100000,50,80
5,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,12:50,80000,70,100
6,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt,14:40,90000,80,100
7,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,17:10,100000,60,80
8,LÊN HƯƠNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh,09:35,90000,80,100
9,LÊN HƯƠNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh,10:20,90000,60,80
